# Import Libraries

In [1]:
import sys
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import sys  # Used in process_single_run to suppress game-by-game output
from io import StringIO # Used in process_single_run to suppress game-by-game output
import warnings
from requests.exceptions import RequestsDependencyWarning
from typing import List, Optional

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Custom imports from the project
from common.utils import load_config, setup_environment_and_agent
from evaluate import play_one_game, evaluate_agent, calculate_summary_stats, find_checkpoints

# Configure for high quality plots
%matplotlib inline
sns.set_theme(style='whitegrid')
%config InlineBackend.figure_format = 'retina'

# Suppress specific warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=".*You are using `torch.load`.*")
warnings.filterwarnings("ignore", category=RequestsDependencyWarning)
print("warnings configured")

c:\Users\Alex\anaconda3\envs\mspacman_rl\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


warnings configured


# Helper Functions

In [ ]:
# Cell 3: Core Analysis Functions

def process_single_run(run_path: Path, num_eval_games: int, target_timesteps: Optional[List[int]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Processes a single experimental run folder, evaluates all its checkpoints,
    and returns the aggregated raw and summary data for that run.

    Parameters:
    - run_path (Path): Path to the run folder (e.g., models/dqn_run_01)
    - num_eval_games (int): Number of games to play per checkpoint for evaluation.
    - target_timesteps (Optional[List[int]]): If provided, only evaluate checkpoints at these timesteps.   

    Returns:
    - tuple: (raw_df, summary_df) DataFrames for this run.
        raw_df: Detailed game-by-game results for all checkpoints in this run.
        summary_df: Aggregated summary statistics per checkpoint in this run.
    """
    #print(f"\n{'='*25} Processing Run: {run_path.name} {'='*25}")
    
    # --- 1. Identify Agent and Load Config ---
    # Infer agent type and config path from the folder structure
    agent_type = run_path.parent.name.split('_')[0].lower()
    config_path = project_root / "configs" / f"{agent_type}_config.yaml"
    
    if not config_path.exists():
        print(f"  [!] Warning: Config file not found at {config_path}. Skipping run.")
        return pd.DataFrame(), pd.DataFrame()
        
    config = load_config(config_path)
    config['agent'] = agent_type
    
    # --- 2. Find Checkpoints ---
    # Use our robust function from evaluate.py
    checkpoint_paths = find_checkpoints(run_path)
    if not checkpoint_paths:
        print(f"  [!] Warning: No checkpoints found. Skipping run.")
        return pd.DataFrame(), pd.DataFrame()
    
    if target_timesteps:
        filtered_paths = []
        target_set = set(target_timesteps)
        for path in checkpoint_paths:
            match = re.search(r'step_(\d+)\.pth$', path.name)
            if match:
                timestep = int(match.group(1))
                if timestep in target_set:
                    filtered_paths.append(path)

        if not filtered_paths:
            print(f"  [!] Warning: No checkpoints matched target timesteps {target_timesteps}. Skipping run.")
            return pd.DataFrame(), pd.DataFrame()
        
        checkpoint_paths_to_process = filtered_paths

    else:
        checkpoint_paths_to_process = checkpoint_paths  # Use all found checkpoints
        
    # --- 3. Setup Environment and Agent (ONCE PER RUN for efficiency) ---
    #print(f"\nSetting up environment and agent for '{agent_type.upper()}'...")
    env, agent, device = setup_environment_and_agent(config)
    
    # --- 4. Loop Through Checkpoints and Evaluate ---
    all_checkpoints_summaries = []
    all_checkpoints_raw_data = []

    checkpoint_prog_bar = tqdm(checkpoint_paths_to_process, desc=f"Evaluating {run_path.name}", leave=False)
    for model_path in checkpoint_prog_bar:
        try:
            timestep = int(re.search(r'step_(\d+)\.pth$', model_path.name).group(1))
            checkpoint_prog_bar.set_postfix_str(f"t={timestep}")
            
            # Load model weights into the EXISTING agent
            agent.load(str(model_path))

            #original_stdout = sys.stdout  # Save a reference to the original standard output
            #sys.stdout = StringIO()       # Redirect standard output to suppress game-by-game output
            
            # --- THIS IS THE CRITICAL CHANGE ---
            # Call our new, clean evaluate_agent function
            raw_df_for_checkpoint = evaluate_agent(agent, env, num_games=num_eval_games, show_progress=False)

            #sys.stdout = original_stdout  # Reset standard output to original

            # Calculate summary stats for this checkpoint
            summary_df_for_checkpoint = calculate_summary_stats(raw_df_for_checkpoint)
            
            # Add metadata for aggregation and plotting
            raw_df_for_checkpoint['timestep'] = timestep
            summary_df_for_checkpoint['timestep'] = timestep
            
            all_checkpoints_raw_data.append(raw_df_for_checkpoint)
            all_checkpoints_summaries.append(summary_df_for_checkpoint)

        except Exception as e:
            #sys.stdout = original_stdout  # Ensure we reset stdout on error
            print(f"  [!] Failed to evaluate checkpoint {model_path.name}. Error: {e}")
            
    # --- 5. Aggregate Data and Clean Up ---
    env.close() # Close the env to free up resources

    if not all_checkpoints_summaries:
        print(f"  [!] No checkpoints were successfully evaluated for {run_path.name}.")
        return pd.DataFrame(), pd.DataFrame()
        
    # Combine all results for this single run
    run_raw_df = pd.concat(all_checkpoints_raw_data, ignore_index=True)
    run_summary_df = pd.concat(all_checkpoints_summaries, ignore_index=True)
    
    # Add run-level metadata
    run_raw_df['run_name'] = run_path.name
    run_summary_df['run_name'] = run_path.name
    run_raw_df['agent_type'] = agent_type
    run_summary_df['agent_type'] = agent_type
    
    #print(f"\n{'='*25} Finished Processing Run: {run_path.name} {'='*25}")
    return run_raw_df, run_summary_df

def run_full_analysis(base_models_dir: Path, num_eval_games: int, target_timesteps: Optional[List[int]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Orchestrates the entire analysis. Discovers all runs, processes them,
    and returns two master DataFrames with all results.

    Parameters:
    - base_models_dir (Path): Path to the base models directory (e.g., models
    - num_eval_games (int): Number of games to play per checkpoint for evaluation.
    - target_timesteps (Optional[List[int]]): If provided, only evaluate checkpoints at these timesteps.

    Returns:
    - tuple: (master_raw_df, master_summary_df)
        master_raw_df: Detailed game-by-game results for all runs and checkpoints.
        master_summary_df: Aggregated summary statistics per run and checkpoint.
    """

    # --- 1. Discover all run paths (using our revised discovery logic) ---
    run_paths = []
    agent_checkpoint_dirs = [p for p in base_models_dir.iterdir() if p.is_dir()]
    for agent_dir in agent_checkpoint_dirs:
        runs_in_dir = [p for p in agent_dir.iterdir() if p.is_dir() and not p.name.startswith('.')]
        run_paths.extend(runs_in_dir)
    run_paths.sort(key=lambda p: p.name)

    if not run_paths:
        print(f"No run folders found in {base_models_dir}. Analysis cannot proceed.")
        return pd.DataFrame(), pd.DataFrame()
    
    print(f"Found {len(run_paths)} total experiment runs to analyze.")
    
    # --- 2. Process each run and collect results ---
    all_runs_raw = []
    all_runs_summaries = []
    
    for path in tqdm(run_paths, desc="Processing Runs"):
        raw_df, summary_df = process_single_run(run_path=path, num_eval_games=num_eval_games, target_timesteps=target_timesteps)
        if not raw_df.empty:
            all_runs_raw.append(raw_df)
            all_runs_summaries.append(summary_df)
            
    # --- 3. Final Aggregation ---
    if not all_runs_raw:
        print("\n\nNo data was successfully processed. Master DataFrames will be empty.")
        return pd.DataFrame(), pd.DataFrame()
        
    master_raw_df = pd.concat(all_runs_raw, ignore_index=True)
    master_summary_df = pd.concat(all_runs_summaries, ignore_index=True)
    
    print("\n\nFull analysis complete. Master DataFrames are ready.")
    return master_raw_df, master_summary_df

In [3]:
def plot_mean_score_vs_steps(summary_df: pd.DataFrame, run_name: str) -> None:
    """
    Plot mean score vs training steps with error bars.

    Parameters:
    - summary_df: DataFrame containing summary statistics indexed by timestep.
    - run_name: Name of the run for labeling the plot.

    Returns:
    - None (displays the plot)
    """
    run_number = summary_df['run_number'].iloc[0] 
    plot_label = f"Run {run_number}"

    plt.figure(figsize=(12, 7))

    plot_df = summary_df.reset_index()

    # Get the data for the plot
    timesteps = plot_df['timestep']
    mean_scores = plot_df['mean_score']
    std_scores = plot_df['std_score']

    plt.plot(timesteps, mean_scores, marker='o', linestyle='-', label=f'{plot_label} (Mean Score)')
    plt.fill_between(timesteps, mean_scores - std_scores, mean_scores + std_scores, alpha=0.2, label=f'{plot_label} (Std Dev)')

    plt.title(f'Agent Performance vs Training Steps ({run_name})', fontsize=16)
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Mean Score', fontsize=12)
    plt.grid(True)
    plt.tight_layout()
    plt.legend()

    plt.show()

def plot_score_distribution_vs_steps(raw_df: pd.DataFrame, run_name: str) -> None:
    """ Creates a box plot of score distributions for each checkpoint.
     
    Parameters:
    - raw_df: DataFrame containing raw evaluation results with 'timestep' and 'score' columns.
    - run_name: Name of the run for labeling the plot.
      
    Returns:
    - None (displays the plot)
    """

    run_number = raw_df['run_number'].iloc[0]

    plt.figure(figsize=(12, 7))
    
    sns.boxplot(x='timestep', y='score', data=raw_df)
    
    plt.title(f'Score Distribution vs. Training Timesteps ({run_name})', fontsize=16)
    plt.xlabel('Training Timesteps', fontsize=12)
    plt.ylabel('Evaluation Score', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.show()

# Run Configuration

In [4]:
# Cell 4: Execute the Full Analysis
TARGET_TIMESTEPS = [
    1000000,
    5000000,
    10000000,
]

# --- Configuration ---
MODELS_DIR = project_root / "models"
NUM_EVAL_GAMES = 10 # Use a small number for testing, increase for final analysis (e.g., 100)

# --- Run the Analysis ---
# This one function call does all the work!
master_raw_df, master_summary_df = run_full_analysis(
    base_models_dir=MODELS_DIR,
    num_eval_games=NUM_EVAL_GAMES,
    target_timesteps=TARGET_TIMESTEPS
)

# --- Verify the Output ---
if not master_summary_df.empty:
    print("\n--- Master Summary DataFrame ---")
    display(master_summary_df.head())
    print("\n--- Unique Runs Analyzed ---")
    print(master_summary_df['run_name'].unique())

Found 6 total experiment runs to analyze.


Processing Runs:   0%|          | 0/6 [00:00<?, ?it/s]

Searching for checkpoints in: g:\Github\DeepRL-MsPacman\models\dqn_checkpoints\DQN_Run_1
Found 15 checkpoints in g:\Github\DeepRL-MsPacman\models\dqn_checkpoints\DQN_Run_1
Using device: cuda
Environment created and wrapped with unified AtariWrapper.
DQN Agent instantiated.


Evaluating DQN_Run_1:   0%|          | 0/3 [00:00<?, ?it/s]

Searching for checkpoints in: g:\Github\DeepRL-MsPacman\models\dqn_checkpoints\DQN_Run_2
Found 20 checkpoints in g:\Github\DeepRL-MsPacman\models\dqn_checkpoints\DQN_Run_2
Using device: cuda
Environment created and wrapped with unified AtariWrapper.
DQN Agent instantiated.


...

Searching for checkpoints in: g:\Github\DeepRL-MsPacman\models\dqn_checkpoints\DQN_Run_6
Found 20 checkpoints in g:\Github\DeepRL-MsPacman\models\dqn_checkpoints\DQN_Run_6
Using device: cuda
Environment created and wrapped with unified AtariWrapper.
DQN Agent instantiated.


Evaluating DQN_Run_6:   0%|          | 0/3 [00:00<?, ?it/s]



Full analysis complete. Master DataFrames are ready.

--- Master Summary DataFrame ---


,mean_score,std_score,min_score,Q1_score,Median_score,Q3_score,max_score,mean_steps,std_steps,level_1_completion_rate,mean_level_reached,max_level_reached,timestep,run_name,agent_type
0,538.0,265.572756,300.0,347.5,410.0,667.5,1120.0,599.2,102.256540,0.0,0.0,0,1000000,DQN_Run_1,dqn
1,1403.0,602.016057,750.0,1165.0,1265.0,1477.5,2980.0,812.0,150.637903,0.0,0.0,0,5000000,DQN_Run_1,dqn
2,2221.0,786.531768,910.0,1747.5,2310.0,2502.5,3540.0,811.8,112.115813,0.0,0.0,0,10000000,DQN_Run_1,dqn
3,1003.0,683.114599,270.0,417.5,860.0,1360.0,2260.0,765.6,230.928657,0.0,0.0,0,1000000,DQN_Run_2,dqn
4,1610.0,357.429092,1080.0,1287.5,1660.0,1870.0,2100.0,816.0,106.003145,0.0,0.0,0,5000000,DQN_Run_2,dqn



--- Unique Runs Analyzed ---
['DQN_Run_1' 'DQN_Run_2' 'DQN_Run_3' 'DQN_Run_4' 'DQN_Run_5' 'DQN_Run_6']


In [5]:
print(master_summary_df.head(20))

    mean_score   std_score  min_score  Q1_score  Median_score  Q3_score  \
0        538.0  265.572756      300.0     347.5         410.0     667.5   
1       1403.0  602.016057      750.0    1165.0        1265.0    1477.5   
2       2221.0  786.531768      910.0    1747.5        2310.0    2502.5   
3       1003.0  683.114599      270.0     417.5         860.0    1360.0   
4       1610.0  357.429092     1080.0    1287.5        1660.0    1870.0   
5       3570.0  804.473603     2490.0    3175.0        3430.0    3717.5   
6        643.0  252.456817      370.0     480.0         535.0     845.0   
7       1449.0  547.853387      960.0    1042.5        1360.0    1567.5   
8       3078.0  965.905447     1960.0    2690.0        2985.0    3087.5   
9        622.0  296.453107      320.0     352.5         555.0     847.5   
10      1443.0  426.225032     1030.0    1152.5        1360.0    1522.5   
11      2909.0  456.665450     2060.0    2695.0        2990.0    3072.5   
12       609.0  251.72515